# Обучение базовых моделей P1P2

In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.precision', 4)

In [2]:
PATH = Path("..", "data", "processed", "p1p2_data.csv")
df = pd.read_csv(PATH, sep=";")

DTTM_COLS = [
    'month',
    'first_trx_month_inn',
    'last_active_month_inn',
    'first_month_p3',
    'first_month_altpay5',
]

for col in DTTM_COLS:
    df[col] = pd.to_datetime(df[col])

# Приводим bool-колонки к int8, чтобы они корректно обрабатывались в моделях
bool_cols = df.select_dtypes(include='bool').columns
for col in bool_cols:
    df[col] = df[col].astype('int8')

## Метрики качества
Основной метрикой качества моделей является Precision@K, поскольку бизнес-задача заключается в эффективном распределении ограниченного ресурса отдела продаж. В рамках задачи capacity отдела составляет 2000 клиентов в месяц, что определяет основное значение K. Дополнительно рассматриваются значения K=500, K=1000, K=1500 и K=2500 для анализа устойчивости качества ранжирования.

Для оценки относительного качества моделей используется метрика Uplift — отношение Precision@K рассматриваемой модели к Precision@K базовых подходов. В качестве базовых подходов используются:

1. Случайный отбор клиентов
2. Отбор на основе текущих бизнес-правил

Дополнительно для оценки качества ранжирования анализируется зависимость Precision@K от K, что позволяет оценить способность модели выделять наиболее релевантных клиентов в верхней части ранжированного списка.

Также используется метрика Recall@K, отражающая долю всех целевых событий (подключений продукта), попадающих в top-K клиентов, что позволяет оценить полноту охвата моделью потенциально релевантных клиентов.

Для интерпретации результатов в бизнес-терминах рассчитывается ожидаемое количество подключений (Expected Conversions), определяемое как произведение Precision@K на K.

## Случайный отбор


Для случайного отбора ожидаемое значение Precision@K совпадает с долей положительных объектов в выборке. В связи с этим в качестве baseline используется доля положительного класса, рассчитанная по каждому месяцу.

In [3]:
from acquiring_xsell.metrics import calculate_random_baseline

print("Random Selection P1P2")

calculate_random_baseline(df=df)

Random Selection P1P2
K=500  | Precision: 0.23% | Coverage: 2.23% | Expected Conversions: 1.15
K=1000 | Precision: 0.23% | Coverage: 4.45% | Expected Conversions: 2.30
K=1500 | Precision: 0.23% | Coverage: 6.68% | Expected Conversions: 3.45
K=2000 | Precision: 0.23% | Coverage: 8.91% | Expected Conversions: 4.59
K=2500 | Precision: 0.23% | Coverage: 11.13% | Expected Conversions: 5.74


## Business-Rule

Критерии релевантности:
- Отсутствие подключённых продуктов `P1`, `P2` (фильтрация уже реализована в `df_p1_p2`)
- Статус `'Active'`, `'Reborn'`
- Группа MCC `!= 'Charity'`

Кандидаты на подключение ранжируется по убыванию `turnover`

In [4]:
from acquiring_xsell.features import add_rule_flag_p1p2

df = add_rule_flag_p1p2(df)

Замерим качество.

In [5]:
from acquiring_xsell.metrics import calculate_metrics

print(f"Business-Rule Selection P1P2, y_score='turnover'")

calculate_metrics(data=df, y_score='turnover', business_rule_mode=True)

Business-Rule Selection P1P2, y_score='turnover'
K=500  | Precision: 0.56% | Recall: 5.67% | Expected Conversions: 2.80
K=1000 | Precision: 0.55% | Recall: 11.33% | Expected Conversions: 5.53
K=1500 | Precision: 0.52% | Recall: 16.06% | Expected Conversions: 7.80
K=2000 | Precision: 0.53% | Recall: 21.24% | Expected Conversions: 10.60
K=2500 | Precision: 0.53% | Recall: 26.61% | Expected Conversions: 13.33


Попробуем улучшить отбор по бизнес-правилу за счёт изменения столбца ранжирования. Переберём все числовые признаки, кроме `target` и `rule_flag`.

In [6]:
numeric_features = [col for col in df.select_dtypes(include=['number']).columns if col not in ('target', 'rule_flag')]

print(f"Business-Rule Selection P1P2\n")

for ascending in (True, False):
    for feature in numeric_features:
        print(f"y_score='{feature}', ascending={ascending}")
        calculate_metrics(
            data=df,
            y_score=feature,
            ascending=ascending,
            business_rule_mode=True
        )
        print()

Business-Rule Selection P1P2

y_score='lifetime_month_streak_inn', ascending=True
K=500  | Precision: 2.79% | Recall: 25.68% | Expected Conversions: 13.93
K=1000 | Precision: 1.84% | Recall: 34.44% | Expected Conversions: 18.40
K=1500 | Precision: 1.40% | Recall: 39.71% | Expected Conversions: 21.00
K=2000 | Precision: 1.13% | Recall: 43.03% | Expected Conversions: 22.67
K=2500 | Precision: 0.99% | Recall: 47.25% | Expected Conversions: 24.80

y_score='real_kam_on_inn', ascending=True
K=500  | Precision: 0.92% | Recall: 9.01% | Expected Conversions: 4.60
K=1000 | Precision: 0.93% | Recall: 17.70% | Expected Conversions: 9.33
K=1500 | Precision: 0.99% | Recall: 27.71% | Expected Conversions: 14.80
K=2000 | Precision: 0.93% | Recall: 34.75% | Expected Conversions: 18.60
K=2500 | Precision: 0.91% | Recall: 42.82% | Expected Conversions: 22.73

y_score='has_termination_start', ascending=True
K=500  | Precision: 0.75% | Recall: 6.92% | Expected Conversions: 3.73
K=1000 | Precision: 0.77% | 

## Base Pipeline

Реализуем универсальный Pipeline обучения моделей для избежания ошибок в расчётах.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

numeric_cols = df.select_dtypes(include='number').columns.tolist()

#pipeline = Pipeline([
#    ('scaler', StandardScaler()),
#    ('model', LogisticRegression())
#])

numeric_cols

['lifetime_month_streak_inn',
 'real_kam_on_inn',
 'has_termination_start',
 'altpay1_turnover',
 'altpay2_turnover',
 'altpay3_turnover',
 'altpay4_turnover',
 'altpay5_turnover',
 'altpay6_turnover',
 'has_recurrent_payments',
 'cnt_trx',
 'turnover',
 'other_active_product_cnt',
 'is_relevant_mcc_p3',
 'is_relevant_mcc_altpay5',
 'is_relevant_turnover_altpay5',
 'active_mcc_cnt',
 'active_gateway_cnt',
 'avg_check',
 'turnover_ma_2m',
 'turnover_ema_2m',
 'turnover_ma_3m',
 'turnover_ema_3m',
 'turnover_ma_6m',
 'turnover_ema_6m',
 'cnt_trx_ma_2m',
 'cnt_trx_ema_2m',
 'cnt_trx_ma_3m',
 'cnt_trx_ema_3m',
 'cnt_trx_ma_6m',
 'cnt_trx_ema_6m',
 'avg_check_wma_2m',
 'avg_check_wma_3m',
 'avg_check_wma_6m',
 'turnover_cv_2m',
 'turnover_cv_3m',
 'turnover_cv_6m',
 'turnover_change_to_prev_month',
 'cnt_trx_change_to_prev_month',
 'avg_check_change_to_prev_month',
 'turnover_change_to_turnover_ma_2m',
 'turnover_change_to_turnover_ma_3m',
 'turnover_change_to_turnover_ma_6m',
 'turnover_ch

## Logistic Regression

Подготовим данные для обучения Логистической регрессии. Обработаем признаки, содержащие месяцы и удалим `inn`, чтобы модель не запоминала данные отдельных клиентов.

In [8]:
from acquiring_xsell.features import add_months_since_last_active_month, add_months_since_first_trx, add_has_p3, add_has_altpay5

df = add_months_since_last_active_month(df)
df = add_months_since_first_trx(df)
df = add_has_p3(df)
df = add_has_altpay5(df)

cols_to_drop = [
    'inn',
    "last_active_month_inn",
    "first_trx_month_inn",
    "first_month_p3",
    "first_month_altpay5",
]

df = df.drop(columns=cols_to_drop)

Разделим датасет на обучающую и тестовую выборки.

In [9]:
train = df[df['month'] < '2025-12-01'].copy() # 80%
test = df[df['month'] >= '2025-12-01'].copy() # 20%

month_train = train['month']  # Сохраняем, т.к. метрики качества усредняются по месяцам (macro average)
X_train = train.drop(columns=['target', 'month'])
y_train = train['target']

month_test = test['month']  # Сохраняем, т.к. метрики качества усредняются по месяцам (macro average)
X_test = test.drop(columns=['target', 'month'])
y_test = test['target']

Применим масштабирование для числовых признаков и One-Hot Encoding для категориальных.

In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

numeric_cols = X_train.select_dtypes(include='number').columns

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

X_train = pd.get_dummies(X_train, drop_first=True)

Обучим логистическую регрессию

In [11]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

lr.fit(X_train, y_train)

y_proba = lr.predict_proba(X_train)[:, 1]

Замерим качество построенной модели

In [12]:
X_train_eval = X_train.copy()
X_train_eval['lr_proba'] = y_proba
X_train_eval[['month', 'target']] = train[['month', 'target']]

print(f"Logistic Regression P1P2 (Train), y_score='lr_proba'")

calculate_metrics(data=X_train_eval, y_score='lr_proba')

Logistic Regression P1P2 (Train), y_score='lr_proba'
K=500  | Precision: 2.82% | Recall: 27.07% | Expected Conversions: 14.08
K=1000 | Precision: 1.80% | Recall: 34.60% | Expected Conversions: 18.00
K=1500 | Precision: 1.39% | Recall: 40.43% | Expected Conversions: 20.92
K=2000 | Precision: 1.15% | Recall: 44.44% | Expected Conversions: 22.92
K=2500 | Precision: 0.97% | Recall: 47.16% | Expected Conversions: 24.33


## LightGBM

## CatBoost